<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Humidity/Humidity_Hybrid_Approach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install river --quiet

In [13]:
import pandas as pd
import numpy as np

from river import metrics, compose, preprocessing

print("River imported successfully")

River imported successfully


In [14]:
import pandas as pd

path = "/content/drive/MyDrive/Research v2/Research 18.03.2026/Humidity/final_humidity_dataset (1).csv"

humidity_df = pd.read_csv(path)

print("Dataset shape:", humidity_df.shape)
humidity_df.head()

Dataset shape: (130003, 16)


,uv_index,cloud,condition_text,air_quality_Ozone,temperature_celsius,feels_like_celsius,air_quality_us-epa-index,air_quality_gb-defra-index,longitude,precip_mm,air_quality_PM10,air_quality_PM2.5,visibility_km,air_quality_Sulphur_dioxide,latitude,humidity
0,1.0,0,2,62.2,16.1,16.1,1,1,-120.49,0.00,7.1,6.3,16.0,0.2,46.60,58
1,1.0,37,32,23.3,23.0,25.3,2,2,-87.22,0.28,25.3,19.0,10.0,1.4,14.10,78
2,1.0,50,23,5.9,26.0,30.2,2,2,-89.20,0.30,28.1,20.4,10.0,7.5,13.71,94
3,1.0,100,19,0.4,20.0,20.0,4,10,-90.53,0.09,178.1,132.0,5.0,19.3,14.62,88
4,1.0,94,30,34.0,26.0,29.6,1,1,-88.77,0.00,32.1,7.7,10.0,0.2,17.25,89


In [15]:
print(humidity_df.columns)

Index(['uv_index', 'cloud', 'condition_text', 'air_quality_Ozone',
       'temperature_celsius', 'feels_like_celsius', 'air_quality_us-epa-index',
       'air_quality_gb-defra-index', 'longitude', 'precip_mm',
       'air_quality_PM10', 'air_quality_PM2.5', 'visibility_km',
       'air_quality_Sulphur_dioxide', 'latitude', 'humidity'],
      dtype='object')


In [17]:
target = "humidity"

In [18]:
# -----------------------------
# HUMIDITY LAG FEATURES
# -----------------------------
humidity_df["humidity_lag1"] = humidity_df[target].shift(1)
humidity_df["humidity_lag2"] = humidity_df[target].shift(2)
humidity_df["humidity_lag3"] = humidity_df[target].shift(3)
humidity_df["humidity_lag5"] = humidity_df[target].shift(5)
humidity_df["humidity_lag7"] = humidity_df[target].shift(7)

# -----------------------------
# ROLLING FEATURES FOR HUMIDITY
# -----------------------------
humidity_df["humidity_roll3_mean"] = humidity_df[target].rolling(window=3).mean()
humidity_df["humidity_roll5_mean"] = humidity_df[target].rolling(window=5).mean()
humidity_df["humidity_roll7_mean"] = humidity_df[target].rolling(window=7).mean()

humidity_df["humidity_roll3_std"] = humidity_df[target].rolling(window=3).std()
humidity_df["humidity_roll5_std"] = humidity_df[target].rolling(window=5).std()

# -----------------------------
# RELATED FEATURE LAGS
# based on humidity heatmap / selected useful features
# -----------------------------
if "temperature_celsius" in humidity_df.columns:
    humidity_df["temp_lag1"] = humidity_df["temperature_celsius"].shift(1)
    humidity_df["temp_lag3"] = humidity_df["temperature_celsius"].shift(3)

if "feels_like_celsius" in humidity_df.columns:
    humidity_df["feels_like_lag1"] = humidity_df["feels_like_celsius"].shift(1)

if "cloud" in humidity_df.columns:
    humidity_df["cloud_lag1"] = humidity_df["cloud"].shift(1)
    humidity_df["cloud_lag3"] = humidity_df["cloud"].shift(3)

if "uv_index" in humidity_df.columns:
    humidity_df["uv_lag1"] = humidity_df["uv_index"].shift(1)

if "precip_mm" in humidity_df.columns:
    humidity_df["precip_lag1"] = humidity_df["precip_mm"].shift(1)
    humidity_df["precip_lag3"] = humidity_df["precip_mm"].shift(3)

if "visibility_km" in humidity_df.columns:
    humidity_df["visibility_lag1"] = humidity_df["visibility_km"].shift(1)

if "air_quality_Ozone" in humidity_df.columns:
    humidity_df["ozone_lag1"] = humidity_df["air_quality_Ozone"].shift(1)

if "air_quality_PM10" in humidity_df.columns:
    humidity_df["pm10_lag1"] = humidity_df["air_quality_PM10"].shift(1)

if "air_quality_PM2.5" in humidity_df.columns:
    humidity_df["pm25_lag1"] = humidity_df["air_quality_PM2.5"].shift(1)

if "air_quality_Sulphur_dioxide" in humidity_df.columns:
    humidity_df["so2_lag1"] = humidity_df["air_quality_Sulphur_dioxide"].shift(1)

# -----------------------------
# SAFE INTERACTION FEATURES
# do NOT use current humidity in interactions
# -----------------------------
if "temperature_celsius" in humidity_df.columns and "cloud" in humidity_df.columns:
    humidity_df["temp_cloud_interaction"] = (
        humidity_df["temperature_celsius"] * humidity_df["cloud"]
    )

if "temperature_celsius" in humidity_df.columns and "precip_mm" in humidity_df.columns:
    humidity_df["temp_precip_interaction"] = (
        humidity_df["temperature_celsius"] * humidity_df["precip_mm"]
    )

if "cloud" in humidity_df.columns and "precip_mm" in humidity_df.columns:
    humidity_df["cloud_precip_interaction"] = (
        humidity_df["cloud"] * humidity_df["precip_mm"]
    )

if "uv_index" in humidity_df.columns and "cloud" in humidity_df.columns:
    humidity_df["uv_cloud_interaction"] = (
        humidity_df["uv_index"] * humidity_df["cloud"]
    )

if "visibility_km" in humidity_df.columns and "precip_mm" in humidity_df.columns:
    humidity_df["visibility_precip_interaction"] = (
        humidity_df["visibility_km"] * humidity_df["precip_mm"]
    )

# -----------------------------
# OPTIONAL: ENCODE condition_text
# -----------------------------
if "condition_text" in humidity_df.columns:
    humidity_df["condition_text"] = humidity_df["condition_text"].astype("category").cat.codes

# -----------------------------
# REMOVE NA CREATED BY LAGS
# -----------------------------
humidity_df = humidity_df.dropna().reset_index(drop=True)

print("After feature engineering:", humidity_df.shape)
humidity_df.head()

After feature engineering: (129989, 44)


,uv_index,cloud,condition_text,air_quality_Ozone,temperature_celsius,feels_like_celsius,air_quality_us-epa-index,air_quality_gb-defra-index,longitude,precip_mm,...,visibility_lag1,ozone_lag1,pm10_lag1,pm25_lag1,so2_lag1,temp_cloud_interaction,temp_precip_interaction,cloud_precip_interaction,uv_cloud_interaction,visibility_precip_interaction
0,1.0,0,2,26.5,27.0,30.0,1,1,-62.72,0.00,...,10.0,21.8,4.3,1.4,0.1,0.0,0.000,0.0,0.0,0.00
1,1.0,25,32,20.2,28.0,32.6,1,1,-61.75,0.06,...,10.0,26.5,4.1,1.0,0.1,700.0,1.680,1.5,25.0,0.60
2,1.0,50,32,20.6,24.0,26.4,1,1,-58.17,0.08,...,10.0,20.2,12.0,2.4,0.7,1200.0,1.920,4.0,50.0,0.64
3,1.0,100,4,1.0,23.1,25.7,1,1,-58.17,0.04,...,8.0,20.6,1.6,1.1,2.9,2310.0,0.924,4.0,100.0,0.00
4,1.0,50,32,0.5,23.0,25.3,1,1,-69.90,0.00,...,0.0,1.0,0.5,0.5,0.1,1150.0,0.000,0.0,50.0,0.00


In [6]:
selected_features = [col for col in humidity_df.columns if col != target]

print("Number of features:", len(selected_features))
print(selected_features)

Number of features: 43
['uv_index', 'cloud', 'condition_text', 'air_quality_Ozone', 'temperature_celsius', 'feels_like_celsius', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'longitude', 'precip_mm', 'air_quality_PM10', 'air_quality_PM2.5', 'visibility_km', 'air_quality_Sulphur_dioxide', 'latitude', 'humidity_lag1', 'humidity_lag2', 'humidity_lag3', 'humidity_lag5', 'humidity_lag7', 'humidity_roll3_mean', 'humidity_roll5_mean', 'humidity_roll7_mean', 'humidity_roll3_std', 'humidity_roll5_std', 'temp_lag1', 'temp_lag3', 'feels_like_lag1', 'cloud_lag1', 'cloud_lag3', 'uv_lag1', 'precip_lag1', 'precip_lag3', 'visibility_lag1', 'ozone_lag1', 'pm10_lag1', 'pm25_lag1', 'so2_lag1', 'temp_cloud_interaction', 'temp_precip_interaction', 'cloud_precip_interaction', 'uv_cloud_interaction', 'visibility_precip_interaction']


In [7]:
split_index = int(0.8 * len(humidity_df))

train_df = humidity_df.iloc[:split_index].copy()
test_df = humidity_df.iloc[split_index:].copy()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (103996, 44)
Test shape : (26000, 44)


In [8]:
n_train = len(train_df)

iter1 = train_df.iloc[:int(0.33 * n_train)].copy()
iter2 = train_df.iloc[int(0.33 * n_train):int(0.66 * n_train)].copy()
iter3 = train_df.iloc[int(0.66 * n_train):].copy()

print("Iteration 1 shape:", iter1.shape)
print("Iteration 2 shape:", iter2.shape)
print("Iteration 3 shape:", iter3.shape)

Iteration 1 shape: (34318, 44)
Iteration 2 shape: (34319, 44)
Iteration 3 shape: (35359, 44)


In [9]:
try:
    from river import forest

    river_model = forest.ARFRegressor(
        n_models=20,
        max_features="sqrt",
        lambda_value=6,
        seed=42
    )
    model_name = "River ARF Hybrid"

except Exception:
    from river import tree

    river_model = compose.Pipeline(
        preprocessing.StandardScaler(),
        tree.HoeffdingAdaptiveTreeRegressor(
            grace_period=50,
            delta=1e-5,
            leaf_prediction="adaptive"
        )
    )
    model_name = "River HAT Hybrid"

print("Using model:", model_name)

Using model: River ARF Hybrid


In [10]:
def run_hybrid_iteration(data, model, iteration_name, selected_features, target, warmup=False):
    mse = metrics.MSE()
    rmse = metrics.RMSE()
    mae = metrics.MAE()
    r2 = metrics.R2()

    y_true_all = []
    y_pred_all = []

    first_part = int(0.1 * len(data)) if warmup else 0

    for i, (_, row) in enumerate(data.iterrows()):
        x = row[selected_features].to_dict()
        y = row[target]

        if i < first_part:
            model.learn_one(x, y)
            continue

        y_pred = model.predict_one(x)
        if y_pred is None:
            y_pred = 0.0

        y_true_all.append(y)
        y_pred_all.append(y_pred)

        mse.update(y, y_pred)
        rmse.update(y, y_pred)
        mae.update(y, y_pred)
        r2.update(y, y_pred)

        model.learn_one(x, y)

    print(f"\n{iteration_name}")
    print("MSE :", round(mse.get(), 4))
    print("RMSE:", round(rmse.get(), 4))
    print("MAE :", round(mae.get(), 4))
    print("R2  :", round(r2.get(), 4))
    print("Accuracy (%):", round(r2.get() * 100, 2))

    return model, mse.get(), rmse.get(), mae.get(), r2.get(), y_true_all, y_pred_all

In [19]:
river_results = []

river_model, mse1, rmse1, mae1, r21, y_true1, y_pred1 = run_hybrid_iteration(
    iter1, river_model, "Iteration 1", selected_features, target, warmup=True
)

river_results.append([
    "Iteration 1", model_name, mse1, rmse1, mae1, r21, r21 * 100
])


Iteration 1
MSE : 129.6154
RMSE: 11.3849
MAE : 8.9467
R2  : 0.7933
Accuracy (%): 79.33


In [20]:
river_model, mse2, rmse2, mae2, r22, y_true2, y_pred2 = run_hybrid_iteration(
    iter2, river_model, "Iteration 2", selected_features, target, warmup=False
)

river_results.append([
    "Iteration 2", model_name, mse2, rmse2, mae2, r22, r22 * 100
])


Iteration 2
MSE : 130.3454
RMSE: 11.4169
MAE : 8.9625
R2  : 0.7658
Accuracy (%): 76.58


In [21]:
river_model, mse3, rmse3, mae3, r23, y_true3, y_pred3 = run_hybrid_iteration(
    iter3, river_model, "Iteration 3", selected_features, target, warmup=False
)

river_results.append([
    "Iteration 3", model_name, mse3, rmse3, mae3, r23, r23 * 100
])


Iteration 3
MSE : 104.9446
RMSE: 10.2442
MAE : 7.9302
R2  : 0.8129
Accuracy (%): 81.29


In [22]:
river_results_df = pd.DataFrame(
    river_results,
    columns=["Iteration", "Model", "MSE", "RMSE", "MAE", "R2", "Accuracy (%)"]
).round(3)

river_results_df


,Iteration,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,Iteration 1,River ARF Hybrid,129.615,11.385,8.947,0.793,79.331
1,Iteration 2,River ARF Hybrid,130.345,11.417,8.963,0.766,76.580
2,Iteration 3,River ARF Hybrid,104.945,10.244,7.930,0.813,81.289


In [23]:
test_mse = metrics.MSE()
test_rmse = metrics.RMSE()
test_mae = metrics.MAE()
test_r2 = metrics.R2()

test_true = []
test_pred = []

for _, row in test_df.iterrows():
    x = row[selected_features].to_dict()
    y = row[target]

    y_hat = river_model.predict_one(x)
    if y_hat is None:
        y_hat = 0.0

    test_true.append(y)
    test_pred.append(y_hat)

    test_mse.update(y, y_hat)
    test_rmse.update(y, y_hat)
    test_mae.update(y, y_hat)
    test_r2.update(y, y_hat)

print("\nFinal Test Performance")
print("MSE :", round(test_mse.get(), 4))
print("RMSE:", round(test_rmse.get(), 4))
print("MAE :", round(test_mae.get(), 4))
print("R2  :", round(test_r2.get(), 4))
print("Accuracy (%):", round(test_r2.get() * 100, 2))


Final Test Performance
MSE : 138.3739
RMSE: 11.7632
MAE : 9.0576
R2  : 0.7113
Accuracy (%): 71.13


In [24]:
river_final_test_df = pd.DataFrame([{
    "Model": model_name,
    "MSE": round(test_mse.get(), 3),
    "RMSE": round(test_rmse.get(), 3),
    "MAE": round(test_mae.get(), 3),
    "R2": round(test_r2.get(), 3),
    "Accuracy (%)": round(test_r2.get() * 100, 3)
}])

river_final_test_df

,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,River ARF Hybrid,138.374,11.763,9.058,0.711,71.129


In [25]:
from google.colab import files

river_results_df.to_csv("river_hybrid_temp_iteration_results.csv", index=False)
river_final_test_df.to_csv("river_hybrid_temp_final_test_results.csv", index=False)

files.download("river_hybrid_temp_iteration_results.csv")
files.download("river_hybrid_temp_final_test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>